# The Arm You Did Not Run

Four topics have chosen a retrieval architecture by reading a table of measured quality and cost.
This notebook walks the topic that takes the table away.

It imports `rag_architecture_bandit_exploration.py`, which owns every number here and on the topic
page. Nothing about retrieval is recomputed: the six architectures, their per-query outcomes and
their costs arrive through the import chain. The only new object is the **feedback structure** —
what a deployment is allowed to observe.

In [ ]:
import pathlib
import sys

import numpy as np

sys.path.insert(0, str(pathlib.Path.cwd() if (pathlib.Path.cwd() / "rag_architecture_bandit_exploration.py").exists()
                       else pathlib.Path.cwd() / "notebooks" / "rag-architecture-bandit-exploration"))
import rag_architecture_bandit_exploration as B

tbl = B.reward_table()
print(f"arms      : {', '.join(B.ARM_NAMES)}")
print(f"classes   : {', '.join(B.LOCAL_CLASSES)}")
print(f"the table : {tbl['ok'].shape[0]} arms x {tbl['ok'].shape[1]} queries")
print(f"deployment: {B.viz_constants()['horizon']} queries, averaged over {B.N_DEPLOY} of them")

## 1. What a deployment cannot see

The full outcome table exists — this notebook can print it — but a deployment never does. It runs
one architecture per query and observes that row only. The other five entries in each column are
counterfactual and stay that way.

The target also moves: as the bridge share of the traffic rises, the architecture that is genuinely
best changes.

In [ ]:
for f in (0.20, 0.35, 0.50, 0.70):
    u = B.true_utility(f)
    best = B.ARM_NAMES[int(np.argmax(u))]
    print(f"  bridge share {f:.2f}:  best = {best:<10s}  utilities = {np.round(u, 3)}")
horizon = B.viz_constants()["horizon"]
total = horizon * len(B.ARM_NAMES)
print()
print(f"over {horizon} queries a deployment collects {horizon} observations")
print(f"out of the {total} outcomes that exist -- one in {len(B.ARM_NAMES)}.")

## 2. Learning the table

Four policies on the same deployments. Regret is measured against the best **fixed** architecture
in hindsight — the object every previous topic in the arc was optimizing.

The last column is the surprise.

In [ ]:
print(f"{'policy':>16s} {'regret':>9s} {'spread':>8s} {'switches':>9s}  beats the fixed arm")
for r in B.policy_table():
    print(f"{r['policy']:>16s} {r['regret']:9.1f} {r['std']:8.1f} {r['switches']:9.1f}  {r['beats_fixed']}/{r['n']}")
d = B.dynamic_vs_static()
print()
print(f"the best fixed arm is `{d['best_fixed_arm']}`; choosing the best arm at every moment")
print(f"instead would have been worth {d['static_gap']:.1f} more.")

Negative regret against the best arm in hindsight is not a policy beating an oracle. It is a
**weak benchmark**: the workload drifts across a boundary, so no single architecture is right for
the whole deployment, and a policy that keeps measuring collects part of that gap by following the
change. The fixed-arm framing the arc optimized was leaving it uncollected.

## 3. The previous topic's instrument, where its premise fails

The fourth topic's whole apparatus was a scalar — how much better must a challenger look before the
incumbent is displaced — and it had a clean interior optimum. Applied here it does not.

In [ ]:
print(f"{'margin':>8s} {'regret':>9s} {'se':>7s} {'switches':>9s}")
for r in B.margin_sweep():
    print(f"{r['margin']:8.3f} {r['regret']:9.1f} {r['se']:7.1f} {r['switches']:9.1f}")
m = B.margins_within_noise()
print()
print(f"{m['n_tied']} of {m['n_total']} settings are within one standard error of the best of them.")
print(f"the margin rule's spread across deployments is {m['spread_ratio']:.2f}x an uncertainty-weighted policy's.")

A constant margin cannot express *how well each arm is currently known*, and that is the only
thing worth knowing once the table is unknown. In the previous topic every arm's estimate refreshed
every window; here an arm's estimate refreshes only when that arm is run.

Paired against UCB on the same deployments — which cancels the shared difficulty of a traffic draw,
the discipline `significance-testing-calibration` established and whose test is imported here:

In [ ]:
for r in B.paired_against_ucb():
    verdict = "significant" if r["p"] < 0.05 else "NOT significant"
    print(f"{r['policy']:>16s}  diff {r['diff']:+8.1f}  t {r['t']:+7.2f}  p {r['p']:.2e}   {verdict}")

The last row is the honest one. A sliding window is the standard remedy for a bandit whose
target moves, and it is directionally better here — but on a single slow changeover it does not
clear the noise. Reported, not promoted.

## 4. A switching cost is a tax on exploration

Exploring means running an architecture you do not currently believe in, and changing architecture
costs something. Total cost is affine in that price, so the whole sweep is arithmetic on one
simulation rather than a re-run per price.

In [ ]:
print(f"{'price':>7s} {'greedy':>9s} {'ucb':>9s} {'thompson':>10s} {'margin':>9s}   winner")
for r in B.switching_cost_table():
    print(f"{r['cost']:7.2f} {r['greedy']:9.1f} {r['ucb']:9.1f} {r['thompson']:10.1f} {r['margin']:9.1f}   {r['winner']}")
c = B.exploration_crossover(B.COST_FINE)
print()
print(f"switch counts: {dict((k, round(v, 1)) for k, v in c['switches'].items())}")
print(f"{c['winner_at_zero']} wins until a switch costs {c['crossover_cost']}, after which {c['winner_after']} does --")
print("not by knowing more, but by having stopped paying to find out.")

## 5. Why the famous rate cannot be read off this corpus

Bandit regret is usually quoted as sqrt(T), and under a switching cost as T^(2/3). It is tempting
to go looking for that exponent in a simulation.

In [ ]:
r = B.rate_study()
print(f"{'T':>7s} {'UCB regret':>11s}")
for T, v in zip(r["horizons"], r["regret"]):
    print(f"{int(T):7d} {v:11.2f}")
print()
print(f"regret against log T is linear with R^2 = {r['log_r2']:.4f}  <- the instance-dependent rate")
print(f"a power law fitted to the same points returns a slope of {r['loglog_slope']:.3f},")
print("which looks like a confirmation of sqrt(T) and is a fit to a logarithm.")

On a corpus with a fixed gap between the arms, UCB's regret is logarithmic — the Lai–Robbins
result, and as good as any policy can do. The sqrt(T) and T^(2/3) rates are **minimax**: worst cases
over instances whose gaps shrink with the horizon so the problem stays hard. One corpus has one gap
and cannot exhibit them, so they are cited in the topic and deliberately not fitted.

## 6. The assertions

Every claim above is a test, including the ones that limit the claims: that forgetting does *not*
measurably help here, and that the shipped laboratory and topic page still carry the numbers this
module produces today. Both drift guards were verified to fail on injected drift before being
trusted.

In [ ]:
B._run_tests()